# ============================================================================
# PORTFOLIO CHATBOT 
## A chatbot that answers questions about Prakhar Dwivedi's professional background
# ============================================================================

In [1]:
# Import all necessary libraries
from dotenv import load_dotenv  # Load environment variables from .env file
from agents import Agent, Runner, function_tool  # AI agent tools
import os  # For accessing environment variables
import requests  # For making HTTP requests
import gradio as gr  # For creating the chat UI
from pypdf import PdfReader  # For reading PDF files
import numpy as np  # For mathematical operations
from sentence_transformers import SentenceTransformer  # For converting text to embeddings
import sys
import warnings
# Load environment variables from .env file
load_dotenv()

True

# ============================================================================
 SUPPRESS TRACING ERRORS
# ============================================================================

In [2]:
# Suppress stderr output PERMANENTLY (kill all those annoying tracing messages)
class SuppressOutput:
    def write(self, x):
        pass
    def flush(self):
        pass

# Redirect stderr PERMANENTLY
sys.stderr = SuppressOutput()

# Suppress all warnings
warnings.filterwarnings("ignore")

 # ============================================================================
   STEP 1: CONFIGURATION - Set up API connections
 # ============================================================================

In [ ]:
# ============================================================================
# DISABLE TRACING
# ============================================================================



# Disable tracing to avoid API conflicts (must be BEFORE importing agents)
os.environ["AGENTS_DISABLE_TRACING"] = "true"
os.environ["ANTHROPIC_DISABLE_TRACING"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

# Tell the code to use local Ollama instead of OpenAI
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"
os.environ["OPENAI_API_KEY"] = "ollama"

# Get notification settings from .env file
notification_user_id = os.getenv("PUSHOVER_USER")
notification_api_token = os.getenv("PUSHOVER_TOKEN")
notification_api_url = "PUT YOUR OWN PUSHOVER API URL"

# ============================================================================
 STEP 2: NOTIFICATION SYSTEM - Send alerts when user contacts or asks questions
# ============================================================================

In [4]:
def send_notification_alert(message_text):
    """
    Send a push notification using Pushover service
    
    Args:
        message_text (str): The message to send
    """
    try:
        # Prepare the data to send
        notification_payload = {
            "user": notification_user_id,
            "token": notification_api_token,
            "message": message_text
        }
        # Send the notification (ignore if it fails)
        requests.post(notification_api_url, data=notification_payload, timeout=5)
    except:
        # If notification fails, continue anyway (non-blocking)
        pass

# ============================================================================
 STEP 3: TOOL 1 - Save user contact information
# ============================================================================

In [5]:
@function_tool
def save_user_contact(email: str, phone_number: str = 'Not provided', name: str = "Not provided", notes: str = "No notes"):
    """
    Save a user's contact details and send a notification
    
    Args:
        email (str): User's email address (required)
        phone_number (str): User's phone number (optional, default: 'Not provided')
        name (str): User's name (optional, default: 'Not provided')
        notes (str): Any additional notes (optional, default: 'No notes')
    
    Returns:
        dict: Status message confirming the contact was saved
    """
    # Create a notification message with all contact details
    notification_message = f"New contact: {name}, Email: {email}, Phone: {phone_number}, Notes: {notes}"
    
    # Send the notification
    send_notification_alert(notification_message)
    
    # Return success message
    return {"status": "Contact saved successfully!"}

# ============================================================================
 STEP 4: TOOL 2 - Save questions the chatbot couldn't answer
# ============================================================================

In [6]:
@function_tool
def save_unanswered_question(question_text: str):
    """
    Save questions that the chatbot couldn't answer for manual review
    
    Args:
        question_text (str): The question the user asked
    
    Returns:
        dict: Status message confirming the question was saved
    """
    # Create a notification message
    notification_message = f"Unanswered question: {question_text}"
    
    # Send the notification
    send_notification_alert(notification_message)
    
    # Return status message
    return {"status": "Question saved for later"}

# ============================================================================
 STEP 5: DOCUMENT LOADING & CHUNKING - Break resume into searchable pieces
# ============================================================================

In [7]:
def load_and_chunk_documents():
    """
    Load resume PDF and summary file, then break them into chunks for searching
    
    Returns:
        list: List of text chunks (pieces of the resume)
    """
    text_chunks = []
    
    # === Load PDF Resume ===
    print("📄 Loading resume PDF...")
    resume_pdf = PdfReader("resume/linkedin.pdf")
    all_resume_text = ""
    
    # Extract text from every page of the PDF
    for page in resume_pdf.pages:
        extracted_text = page.extract_text()
        if extracted_text:
            all_resume_text += extracted_text
    
    # === Smart Chunking: Split by Resume Sections ===
    # Instead of random chunks, split by meaningful sections (Education, Experience, etc.)
    resume_sections = []
    current_chunk = ""
    
    # Go through each line and detect section headers
    for line in all_resume_text.split('\n'):
        # Check if this line is a section header (Education, Experience, etc.)
        if line.strip() in ['Education', 'Experience', 'Projects', 'Skills', 'Awards & Accomplishments', 'Competitive Conqueror', 'Counselxperts', 'Sawari']:
            # Save the previous chunk
            if current_chunk:
                resume_sections.append(current_chunk.strip())
            # Start a new chunk with this header
            current_chunk = line + "\n"
        else:
            # Add this line to the current chunk
            current_chunk += line + "\n"
    
    # Don't forget the last chunk
    if current_chunk:
        resume_sections.append(current_chunk.strip())
    
    # Add the full resume text as one chunk for broad queries
    resume_sections.append(all_resume_text)
    
    # === Load Summary File ===
    print("📝 Loading summary file...")
    with open("resume/summary.txt", "r", encoding="utf-8") as file:
        summary_text = file.read().strip()
        resume_sections.append(summary_text)
    
    # Log how many chunks we created
    print(f"✓ Created {len(resume_sections)} text chunks\n")
    
    return resume_sections

# ============================================================================
 STEP 6: INITIALIZE RAG (Retrieval-Augmented Generation) SYSTEM
# ============================================================================

In [8]:
print("🚀 Starting up RAG system...")

# Load the embedding model (converts text to numbers for comparison)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Load all resume chunks
all_document_chunks = load_and_chunk_documents()

# Convert each chunk to embeddings (numerical representations)
# This allows us to compare which chunks match user questions
all_chunk_embeddings = embedding_model.encode(all_document_chunks)

print("✓ RAG system ready!\n")

🚀 Starting up RAG system...
📄 Loading resume PDF...
📝 Loading summary file...
✓ Created 8 text chunks

✓ RAG system ready!



# ============================================================================
 STEP 7: SEARCH FUNCTION - Find relevant resume sections for user questions
# ============================================================================

In [9]:
def find_matching_resume_sections(user_question, number_of_results=3):
    """
    Search through resume chunks and find the most relevant ones
    
    Args:
        user_question (str): What the user asked
        number_of_results (int): How many chunks to return (default: 3)
    
    Returns:
        str: The matching resume sections joined together
    """
    # Convert the user's question to embedding (numbers)
    question_embedding = embedding_model.encode([user_question])[0]
    
    # Calculate similarity between question and each chunk
    # (using cosine similarity - how similar vectors are)
    similarity_scores = np.dot(all_chunk_embeddings, question_embedding) / (
        np.linalg.norm(all_chunk_embeddings, axis=1) * np.linalg.norm(question_embedding)
    )
    
    # Get the indices of the top matching chunks
    top_chunk_indices = np.argsort(similarity_scores)[-number_of_results:][::-1]
    
    # Extract the actual text from the top chunks
    matching_chunks = [all_document_chunks[i] for i in top_chunk_indices]
    
    # Join them together with a separator
    return "\n\n---\n\n".join(matching_chunks)

# ============================================================================
 STEP 8: CREATE THE AI AGENT - The "brain" of the chatbot
# ============================================================================

In [10]:
# This is the prompt that tells the AI how to behave
system_instructions = """You are Prakhar Dwivedi's portfolio assistant.

RULES:
1. Answer ONLY from the CONTEXT provided below
2. If the answer is in CONTEXT, give it accurately and professionally
3. If the answer is NOT in CONTEXT, call save_unanswered_question
4. If user provides contact info, call save_user_contact
5. Be concise and professional in your responses
6. You can respond to basic greetings like Hi, Hello, Bye, etc

Use these emojis to be friendly: 🙂🚀👍

---
CONTEXT (Resume information):
{context}
---
"""

# Create the chatbot agent with tools
portfolio_assistant = Agent(
    name="Portfolio Assistant",
    instructions="",  # Will be filled dynamically with context
    model="gpt-oss:120b-cloud",  # Using local Ollama model
    tools=[save_user_contact, save_unanswered_question]  # Tools available to the agent
)

# ============================================================================
 STEP 9: CHAT FUNCTION - Handle user messages and generate responses
# ============================================================================

In [ ]:
async def chat_with_assistant(user_message, conversation_history):
    """
    Process a user message and generate a response from the assistant
    
    Args:
        user_message (str): What the user just typed
        conversation_history (list): All previous messages in the chat
    
    Returns:
        str: The assistant's response
    """
    # Step 1: Find relevant resume sections for this question
    relevant_resume_context = find_matching_resume_sections(user_message, number_of_results=5)
    
    # Step 2: Fill in the instructions with the relevant context
    # Thought Process: I want to update my agent's instruction with context(via RAG) from user's message. 
    #                  So that with easch new user input it gets relevant context from RAG pipeline. And changes it instructions. 
    #                  And hence when await Runner.run() is compiled it will have agent with instructions that has relevant context. Plus complete message history.               
    portfolio_assistant.instructions = system_instructions.format(context=relevant_resume_context) # system_instruction has f string of context, which will get updated.
    
    # Step 3: Add the user's new message to the conversation history
    full_messages = conversation_history + [{"role": "user", "content": user_message}]
    
    # Step 4: Run the agent and get the response
    agent_result = await Runner.run(portfolio_assistant, input=full_messages)
    
    # Step 5: Return just the final text response
    return agent_result.final_output

# ============================================================================
 STEP 10: CREATE THE CHAT INTERFACE - The UI that users interact with
# ============================================================================

In [12]:
gr.ChatInterface(
    fn=chat_with_assistant,  # Function to call for each message
    type="messages",  # Use message format (not plain text)
    examples=[
        "Tell me about your projects",
        "What technologies do you work with?",
        "What is your work experience?",
        "What is your college and school name?"
    ]
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
